In [1]:
import logging
from IPython.display import Image, display
from langchain_huggingface import HuggingFaceEmbeddings

# Importar desde los nuevos paquetes de 'core'
from core.rag.vector_db import DualCorpusManager
from core.agentic.core import AgentOrchestrator

# ==========================================
# 0. Configuración de Logging
# ==========================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("AgentNotebook")

# ==========================================
# 1. Inicialización de Componentes
# ==========================================
logger.info("Inicializando modelo de Embeddings y Vector DB...")
try:
    # Asegúrate de que la ruta al modelo y a la base de datos sea correcta
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    db_manager = DualCorpusManager(embeddings_model=embeddings, persist_directory="../data/chroma/local_chroma_db")
    db_manager.load_existing_db()
    logger.info("Vector DB cargada exitosamente.")
except Exception as e:
    logger.error(f"Error inicializando componentes de RAG: {e}", exc_info=True)
    raise

# ==========================================
# 2. Instanciación del Orquestador
# ==========================================
logger.info("Instanciando el orquestador del agente...")
try:
    # La ruta a las credenciales es relativa a la raíz del proyecto donde se ejecuta el notebook
    agent_orchestrator = AgentOrchestrator(db_manager=db_manager, credentials_path="../.cred/credentials.yaml")
    logger.info("Orquestador del agente creado exitosamente.")
except Exception as e:
    logger.error(f"Error instanciando AgentOrchestrator: {e}", exc_info=True)
    raise

# ==========================================
# 3. Ejecución del Agente
# ==========================================
# Transcripción de prueba para la PoC
input_state = {
    "transcription": (
        "Visita de seguimiento con la doctora Elena Torres. Le he llevado la ficha técnica "
        "de Glucofast que me pidió la semana pasada. Hemos estado revisando juntos el apartado "
        "de posología. Le he subrayado que, según la ficha, para el segmento de pacientes con "
        "insuficiencia renal moderada, es decir, con un filtrado entre 30 y 60, la indicación de "
        "la marca es ajustar a 25 miligramos al día."
    )
}

logger.info("Iniciando la ejecución del agente LangGraph...")
final_state = agent_orchestrator.invoke(input_state)

# ==========================================
# 4. Visualización de Resultados
# ==========================================
print("\n" + "="*30)
print("=== RESULTADO FINAL DE LA PoC ===")
print("="*30)
print(f"Compliance Validado: {final_state.get('is_compliant', 'N/A')}")
print(f"Razonamiento del Auditor: {final_state.get('compliance_reasoning', 'N/A')}\n")
print("Recomendación del Orquestador:")
print(final_state.get('final_recommendation', 'No se generó ninguna recomendación.'))
print("="*30 + "\n")

# ==========================================
# 5. Visualización del Grafo (Opcional)
# ==========================================
logger.info("Intentando renderizar el grafo del agente...")
try:
    # El método get_graph() debe estar disponible en tu instancia compilada
    graph_image = agent_orchestrator.graph.get_graph().draw_mermaid_png()
    display(Image(graph_image))
    logger.info("Grafo renderizado y mostrado en el notebook.")
except Exception as e:
    logger.error(f"No se pudo renderizar el grafo. Asegúrate de que las dependencias (ej. pygraphviz) estén instaladas: {e}")


2026-08-07 01:41:48 [INFO] AgentNotebook: Inicializando modelo de Embeddings y Vector DB...
2026-08-07 01:41:49 [INFO] numexpr.utils: NumExpr defaulting to 12 threads.
2026-08-07 01:41:51 [INFO] sentence_transformers.base.model: No device provided, using cuda:0
2026-08-07 01:41:51 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-07 01:41:51 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-07 01:41:51 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"
2026-08-07 01:41:51 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-mul

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-07 01:41:53 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-08-07 01:41:53 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-07 01:41:53 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-07 01:41:54 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-07 01:41:54 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redire


=== RESULTADO FINAL DE LA PoC ===
Compliance Validado: True
Razonamiento del Auditor: La dosis recomendada por el delegado coincide con la indicación de la ficha técnica para pacientes con insuficiencia renal moderada

Recomendación del Orquestador:
- Enviar un correo electrónico de seguimiento a la doctora Elena Torres con información adicional sobre el manejo de Glucofast en pacientes con insuficiencia renal moderada
- Agendar una visita adicional para discutir casos de estudio de pacientes que se han beneficiado del ajuste de dosis de Glucofast
- Preparar materiales educativos personalizados para el equipo de la doctora Torres sobre el uso óptimo de Glucofast en diferentes escenarios clínicos



2026-08-07 01:42:22 [ERROR] AgentNotebook: No se pudo renderizar el grafo. Asegúrate de que las dependencias (ej. pygraphviz) estén instaladas: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
